# Mapping the Workforce: A Social Network Analysis of Job Interconnectivity
## O*NET to Gephi — Pearson Correlation Approach

**Edge Definition**: Two occupations are linked if the Pearson correlation between their skill importance score profiles exceeds 90% similarity
> **Intuition:** If two jobs rank their skills in a similar order of importance, they are connected.
> *"These occupations value the same skills with the same level of priority."*

### Required Files
Download from [O\*NET 30.1](https://www.onetcenter.org/database.html) as **Text/Tab-delimited** (**Note:** Data will be aggregated between these two data files):
- `Occupation Data.txt`
- `Skills.txt`
-----------------------------------------------------
- `fo_data.csv`- Data from Frey & Osborne's study: "The future of employment: How susceptible are jobs to computerisation?"
        -  https://www.sciencedirect.com/science/article/abs/pii/S0040162516302244
- `crosswalk.csv` - 2010 to 2018 crosswalk conversion file
        -  https://www.bls.gov/soc/2018/soc_2010_to_2018_crosswalk.xlsx

### Outputs
- `remaining_nodes.csv` — Node table for Gephi with a set percentage of routine jobs removed
- `remainng_edges.csv` — Edge table for Gephi with a set percentage of routine jobs removed

### Tools
Google Gemini was used for error correction and code restructuring

### 1. Imports

In [12]:
import pandas as pd
import numpy as np
import networkx as nx
from itertools import combinations

### 2. Initialization

In [13]:
OCCUPATION_FILE = "Occupation Data.txt"
SKILLS_FILE     = "Skills.txt"
THRESHOLD       = 0.95   # Pearson correlation cutoff (-1 to 1)
SCALE           = "IM"   # Select only IMPORTANCE values

### 3. Load Data from Files

In [14]:
def load_data():
    occ = pd.read_csv(OCCUPATION_FILE, delimiter="\t", usecols=["O*NET-SOC Code", "Title"]) # Read data from occupation file
    skills = pd.read_csv(SKILLS_FILE, delimiter="\t",usecols=["O*NET-SOC Code", "Element ID", "Scale ID", "Data Value"]) # Read data from skills file
    return occ, skills

occ, skills = load_data()
occ.head() # Display the first 5 rows

,O*NET-SOC Code,Title
0,11-1011.00,Chief Executives
1,11-1011.03,Chief Sustainability Officers
2,11-1021.00,General and Operations Managers
3,11-1031.00,Legislators
4,11-2011.00,Advertising and Promotions Managers


### 4. Build Matrix

In [15]:
def build_matrix(skills):
    filtered = skills[skills["Scale ID"] == SCALE] # Filter skills where the Scale ID is only IMPORTANCE values

    matrix = filtered.pivot_table(
        index="O*NET-SOC Code",
        columns="Element ID",
        values="Data Value",
        aggfunc="mean"
    ).fillna(0) # Create a matrix by pivoting the skills array so that each skill SOC code and Element ID column has a data value.
    return matrix

matrix = build_matrix(skills) # Build the matrix by calling function
matrix.head() # Retrieve the top 5 entries

Element ID,2.A.1.a,2.A.1.b,2.A.1.c,2.A.1.d,2.A.1.e,2.A.1.f,2.A.2.a,2.A.2.b,2.A.2.c,2.A.2.d,...,2.B.3.k,2.B.3.l,2.B.3.m,2.B.4.e,2.B.4.g,2.B.4.h,2.B.5.a,2.B.5.b,2.B.5.c,2.B.5.d
O*NET-SOC Code,,,,,,,,,,,,,,,,,,,,,
11-1011.00,4.12,4.00,4.12,4.25,3.25,1.62,4.38,3.75,3.12,4.00,...,1.50,1.0,1.88,4.75,4.12,4.25,4.00,4.25,4.00,4.25
11-1011.03,4.00,4.00,4.12,4.00,2.88,2.12,4.12,3.75,3.38,3.75,...,1.00,1.0,1.88,3.88,3.88,3.88,3.38,2.88,2.25,3.12
11-1021.00,4.00,4.00,3.50,4.00,2.62,1.50,3.88,3.62,3.00,4.00,...,1.75,1.0,2.38,3.62,3.12,3.12,3.62,3.00,3.12,3.75
11-2011.00,3.75,4.12,3.75,4.00,3.00,1.62,4.00,3.25,3.00,3.25,...,1.00,1.0,1.62,3.75,3.12,3.12,3.50,2.75,2.62,3.12
11-2021.00,3.88,3.88,3.25,3.88,2.75,1.75,3.88,3.88,3.12,3.75,...,1.00,1.0,1.88,3.75,3.25,3.50,3.50,2.88,2.62,3.38


### 5. Compute Edges using Pearson Correlation

In [16]:
def compute_edges(matrix):

    corr_matrix = matrix.T.corr() # First transpose the skill matrix, then compute pearson correlation
    codes = corr_matrix.index.tolist() # Retrieve the columns of the matrix as a list
    edges = [] # Initialize an array for edges

    # Loop through the occupations and check every unique pair of occupations
    for i in range(len(codes)):
        for j in range(i + 1, len(codes)):
            # Retrieve the correlation value between occupation i and j
            corr = corr_matrix.iloc[i, j]

            # Only keep pairs whose pearson correlation is above or equal to set threshold
            if corr >= THRESHOLD:
                # Add the entry to the edge list when condition met
                edges.append({
                    "Source": codes[i],
                    "Target": codes[j],
                    "Weight": round(float(corr), 4)
                })

    edges_df = pd.DataFrame(edges) # Create a dataframe using the edges
    return edges_df

edges_df = compute_edges(matrix) # Compute the edges using the previously created matrix
print(f"Edges: {len(edges_df)}")

Edges: 3071


### 6. Nodes Table

In [17]:
def build_nodes(occ, edges_df):
    connected = set(edges_df["Source"]).union(set(edges_df["Target"])) # Collect all occupation that appears in the edge dataframe (use set to only collect unique occupations)
    nodes_df = occ[occ["O*NET-SOC Code"].isin(connected)].copy() # Filter occupations from the dataset to only keep the occupations that are within the edges dataframe
    nodes_df = nodes_df.rename(columns={"O*NET-SOC Code": "Id", "Title": "Label"}) # Rename columns as expected by Gephi

    nodes_df["SOC_Major"] = nodes_df["Id"].str[:2] # Extract the first two characters of the occupation code

    soc_labels = { # Dictionary of each occupation code and label (Retrieved from https://www.onetcenter.org/database.html)
        "11": "Management",
        "13": "Business & Financial",
        "15": "Computer & Math",
        "17": "Architecture & Engineering",
        "19": "Life, Physical & Social Science",
        "21": "Community & Social Service",
        "23": "Legal",
        "25": "Education",
        "27": "Arts & Media",
        "29": "Healthcare Practitioners",
        "31": "Healthcare Support",
        "33": "Protective Service",
        "35": "Food Preparation",
        "37": "Building & Grounds",
        "39": "Personal Care",
        "41": "Sales",
        "43": "Office & Admin Support",
        "45": "Farming, Fishing & Forestry",
        "47": "Construction & Extraction",
        "49": "Installation & Repair",
        "51": "Production",
        "53": "Transportation",
        "55": "Military"
    }
    nodes_df["SOC_Group"] = nodes_df["SOC_Major"].map(soc_labels).fillna("Other") # Map each SOC label to each occupation
    return nodes_df # Return the nodes dataframe

nodes_df = build_nodes(occ, edges_df) # Build the nodes
print(f"Nodes: {len(nodes_df)}")

Nodes: 525


### 7. Remove % of Routine Jobs

In [18]:
fo_df = pd.read_csv("fo_data.csv")
cw_df = pd.read_csv("crosswalk.csv")

# Data Normalization Step
# Citation: pandas Vectorized String Operations
# https://pandas.pydata.org/docs/reference/api/pandas.Series.str.split.html
nodes_df['clean_id'] = nodes_df['Id'].str.split('.').str[0].str.strip()
cw_df['2010 SOC Code'] = cw_df['2010 SOC Code'].astype(str).str.strip()
cw_df['2018 SOC Code'] = cw_df['2018 SOC Code'].astype(str).str.strip()
fo_df['SOC code'] = fo_df['SOC code'].astype(str).str.strip()

# Link FO scores to the Crosswalk
# pandas Merge/Join
# https://pandas.pydata.org/docs/reference/api/pandas.merge.html
automation_map = pd.merge(cw_df, fo_df, left_on="2010 SOC Code", right_on="SOC code")
print(f"\nSuccessfully mapped {len(automation_map)} automation scores to the crosswalk.")

# Merge with nodes_df; join the automation_map to nodes using the 6-digit codes
nodes_with_risk = pd.merge(
    nodes_df, 
    automation_map[['2018 SOC Code', 'Probability']], 
    left_on="clean_id", 
    right_on="2018 SOC Code", 
    how="left"
)

# Check Join Success
matched_count = nodes_with_risk['Probability'].notna().sum()
print(f"Nodes with assigned Probability: {matched_count} / {len(nodes_df)}")

if matched_count == 0:
    print("No matches found. Check if hyphens exist in one file but not the other (e.g., 11-1011 vs 111011).")

# Clean duplicates
# pandas GroupBy and Aggregation
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html
nodes_with_risk = nodes_with_risk.groupby(['Id', 'Label', 'SOC_Group']).agg({'Probability': 'mean'}).reset_index()

# Identify Jobs to remove
# pandas Quantile
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.quantile.html
percentile_threshold = nodes_with_risk['Probability'].quantile(0.524)
print(f"\n80th Percentile Probability Threshold: {percentile_threshold:.4f}")

automatable_jobs = nodes_with_risk[nodes_with_risk['Probability'] >= percentile_threshold]

print(f"Number of jobs identified as Highly Automatable: {len(automatable_jobs)}")


Successfully mapped 715 automation scores to the crosswalk.
Nodes with assigned Probability: 446 / 525

80th Percentile Probability Threshold: 0.3004
Number of jobs identified as Highly Automatable: 209


### 8. Export CSV to Gephi

In [19]:
# Identify IDs of the remaining nodes
# take list of nodes and exclude those identified as Highly Automatable
# pandas Boolean Indexing and the Tilde (~) Operator for Inversion:
# https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing
remaining_nodes_df = nodes_with_risk[~nodes_with_risk['Id'].isin(automatable_jobs['Id'])].copy()

# Filter the Edges
# keep an edge only if both the Source and Target occupations are in the remaining_nodes list
remaining_ids = set(remaining_nodes_df['Id'])

# pandas Series.isin:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html
remaining_edges_df = edges_df[
    (edges_df['Source'].isin(remaining_ids)) & 
    (edges_df['Target'].isin(remaining_ids))
].copy()

# Export to CSV
remaining_nodes_df.to_csv("remaining_nodes.csv", index=False)
remaining_edges_df.to_csv("remaining_edges.csv", index=False)

print("--- Simulation Results ---")
print(f"Original Nodes: {len(nodes_df)}")
print(f"Nodes Removed: {len(automatable_jobs)}")
print(f"Nodes Remaining: {len(remaining_nodes_df)}")
print(f"Edges Remaining: {len(remaining_edges_df)}")
print("\nFiles 'remaining_nodes.csv' and 'remaining_edges.csv' are ready.")

--- Simulation Results ---
Original Nodes: 525
Nodes Removed: 209
Nodes Remaining: 316
Edges Remaining: 2135

Files 'remaining_nodes.csv' and 'remaining_edges.csv' are ready.
